# Stage1 分监督域输出头

假设：主ETF与辅助个股合成2x的标签不同，可能需要不同输出映射；保留共享编码器和辅助梯度。B为原共享头匹配重训，C只克隆并分开辅助tail/alpha头。并不预设负迁移已经成立。

冻结设计见[plan.json](plan.json)，原始方案及独立评审变更见[plan_before_review.json](plan_before_review.json)。2023 cutoff共6次拟合，seed71/172/273全部保留、固定300轮；2020辅助0条不跑。

In [1]:
import os, sys, json
from pathlib import Path
root = Path.cwd()
while not (root / "AGENTS.md").exists():
    if root.parent == root:
        raise RuntimeError("workspace root not found")
    root = root.parent
sys.path.insert(0, str(root))
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"
import qlib
from qlib.data import D
from qlib.workflow import R
from src.workflow.stage1_domain_heads import prepare, run
card = root / "research/stage1_domain_heads_20261018"
plan = json.loads((card / "plan.json").read_text())
qlib.init(provider_uri=str(root / plan["provider_init"]), region="us", kernels=1)


## 冻结来源与成熟样本预检

只读哈希绑定的Qlib Recorder，经DatasetH.prepare恢复原输入。主3014行、辅助20741行，39主输入与16共享输入；不读取当前脏provider替换来源。A为现役f696+b472在同源09:45已知→09:47成交→10:32恢复研究合同下重放，不是09:31/10:31运行记录。A/B/C独立费用状态、原路由和恢复，无T/COMMON/ALL3。

In [2]:
prepared = prepare(root=root, card=card)
prepared["receipt"]


{'plan_sha256': '96dc55af2318e26162e7ff8f74c4fdf24bcd15c4d9a3c4e1e2ca18357e3e17c7',
 'primary_rows': 3014,
 'auxiliary_rows': 20741,
 'primary_last_mature': '2022-12-30 09:31:00-05:00',
 'aux_first_signal': '2019-11-25 00:00:00',
 'aux_last_signal': '2022-12-22 00:00:00',
 'aux_observations_end_max': '2022-12-30 00:00:00',
 'source_hashes_checked': True,
 'source_api': ['Recorder.load_object', 'DatasetH.prepare'],
 'main_inputs': 39,
 'common_inputs': 16,
 'account_sessions': 944,
 'early_fields_exact_source_equality': ['early_ret_soxl',
  'early_ret_soxx',
  'early_range_soxl',
  'early_range_soxx',
  'early_from_low_soxl',
  'early_from_low_soxx',
  'early_volume_soxl',
  'early_volume_soxx',
  'early_bar_count_soxl',
  'early_bar_count_soxx'],
 'reused_reference_accounts': 9,
 'active_bundle_and_classifier_hashes_verified': True,
 'initialization': [{'seed': 71,
   'original_parameters': 973,
   'candidate_parameters': 991,
   'original_tensors_identical': True,
   'initial_primary_

## 固定训练、完整账户与分析

正式执行还要求独立design_review与owner execution_approval匹配计划及最终代码hash。分类成绩不决定是否运行原定完整账户。主/辅助初始输出相同，新增18参数以及全局梯度裁剪变化属于整体干预。原价格、公司行动与已完成SOXX/9097/27baa九个原生账户严格同口径后复用；新增A/B/C九账户，0/2/5bp分别结算。

In [3]:
execute = os.environ.get("STAGE1_DOMAIN_HEADS_EXECUTE") == "1"
if execute:
    result = run(root=root, card=card, prepared=prepared)
else:
    result = json.loads((card / "summary.json").read_text())
result


completed fit B_shared_71


completed fit B_shared_172


completed fit B_shared_273


completed fit C_domain_71


completed fit C_domain_172


...

completed native account B_shared_5


completed native account C_domain_0


completed native account C_domain_2


completed native account C_domain_5


{'experiment': 'stage1_domain_heads_20261018',
 'plan_sha256': '96dc55af2318e26162e7ff8f74c4fdf24bcd15c4d9a3c4e1e2ca18357e3e17c7',
 'decision': 'not_promoted_fixed_stage1_domain_heads',
 'objective_achieved': False,
 'gates': {'net_relative_wealth_all_costs': False,
  'sampled_drawdown_30pct': True,
  'cvar_no_worse_soxx': True,
  'minimum_complete_years': True,
  'complete_annual_70pct': False,
  'rolling126_70pct': False,
  'paired_block_lower_positive': False,
  'fresh_fullminute_risk_qualified': False,
  'qualified_independent_forward': False},
 'coarse_economic_gates_pass': False,
 'new_fits': 6,
 'new_native_accounts': 9,
 'reused_native_accounts': 9,
 'total_comparison_accounts': 18,
 'sessions_each': 944,
 'recorder_id': '6e227278b7a64eafbcf34b965e16d2b9',
 'experiment_id': '283510277269468590',
 'recorder_uri': '/Users/rziiiii/dev/workspace/research/stage1_domain_heads_20261018/qlib_recorder',
 'recorder_local_dir': '/Users/rziiiii/dev/workspace/research/stage1_domain_heads_20

In [4]:
from IPython.display import Markdown, display
display(Markdown((card / "conclusion.md").read_text()))


# Stage1分监督域输出头：固定配对实验

完成B共享头与C分域头各3个seed、共6次固定300轮拟合。主3014条、辅助20741条；39主输入与16共享输入、标签、样本权重及原Stage1路由/风险/恢复均固定。新增18参数，仍允许辅助梯度更新共享编码器。

A是现役f696+b472在同源原始股数、09:45已知/09:47成交/10:32恢复研究合同下重放，不是运行中09:31/10:31实盘复现。新增9个A/B/C独立费用原生账户，严格同口径复用SOXX/9097/27baa九账户；每账户944区间。

|方案|滑点bp|累计净收益%|收盘最大回撤%|CVaR5%|
|---|---:|---:|---:|---:|
|A_active_research_replay|0|229.686|-29.487|-4.389|
|A_active_research_replay|2|280.650|-27.060|-4.211|
|A_active_research_replay|5|41.521|-30.517|-4.026|
|B_shared|0|280.383|-25.709|-4.215|
|B_shared|2|270.223|-25.766|-4.215|
|B_shared|5|265.858|-25.819|-4.204|
|C_domain|0|254.630|-27.940|-4.234|
|C_domain|2|187.482|-28.015|-4.330|
|C_domain|5|183.424|-28.052|-4.332|
|SOXX|0|380.588|-40.863|-4.905|
|SOXX|2|380.568|-40.867|-4.905|
|SOXX|5|380.538|-40.872|-4.906|
|strong9097|0|441.681|-26.812|-4.399|
|strong9097|2|434.269|-26.881|-4.400|
|strong9097|5|423.730|-26.956|-4.401|
|additional27baa|0|485.498|-27.171|-4.454|
|additional27baa|2|477.826|-27.223|-4.455|
|additional27baa|5|466.073|-27.317|-4.457|

决定：**not_promoted_fixed_stage1_domain_heads**。
失败或未取得的门：net_relative_wealth_all_costs, complete_annual_70pct, rolling126_70pct, paired_block_lower_positive, fresh_fullminute_risk_qualified, qualified_independent_forward。

全部年度、季度、126交易日窗口与三seed分类结果均保留；不择窗、择seed或按分类分数取消账户。首尾不完整年度单列，复用历史与重叠窗口不构成独立验证。参数分离和全局梯度裁剪/优化器变化共同构成本次干预，不能把差异解释成已证明的唯一负迁移原因。

净收益来自原始股数/现金/分红应收/费用独立账本并经Qlib核对。采样回撤不认证全分钟风险；公开历史来源与纸面公司行动约定不等于历史PIT或真实券商结算。现役生产未改。

关闭本固定分域头版本，不调整head数量、损失系数、epoch、seed、窗口或控制器救援；整体稳定超额目标仍未实现。

Qlib Recorder：`6e227278b7a64eafbcf34b965e16d2b9`；来源与NAS身份见summary.json、verification_manifest.json、nas_binding.json。


## 独立评估与一次必要归因

独立评估接受固定负结论，核验6次拟合、9个新原生账户与9个复用账户身份及全部门禁；B三个seed与冻结匹配来源的网络张量/预测最大差为0。12项所需配对置信下界全部为负，完整2024/2025两个年度各成本均跑输SOXX。见independent_evaluation.json。

主alpha的BCE/Brier由0.68788/0.24740降到0.68346/0.24532，但AUC由0.49690到0.49675；预测均值由52.31%到56.91%，接近57.13%事件率。这支持局部校准改善，未支持新增时序排序。tail的AUC略改善而Brier由0.09957恶化到0.10059。

2bp候选平滑原始提议beta约1.0723，高于匹配对照1.0666；实际平均beta却为0.7938，对照0.8491，现金48.66%对46.43%。非零风险状态545日对377日，均为各自账户的内生路径，不能把收益差单独归因于一个控制器。两账户944区间/3778点原生对账误差为0。候选0bp与2bp有99日风险状态不同，成本敏感性不等于简单从同一路径扣费。见failure_attribution.json；没有为归因新增拟合或回放。

整体Goal仍为费用后稳定超过SOXX，未完成。下一问题是寻找能区分未来相对收益条件的信息，而非只拟合总体事件率。按用户新增要求，量化社区/社交平台作为发现线索，原论文、代码和官方数据作为核查依据；当前启动FINRA场外short标记成交结构的来源资格与独立设计，尚不声称其有效，也不重开本分域头。
负责人从NAS冷恢复并核验94个Recorder对象，全部SHA一致；原生Qlib读回summary门禁和完成计数相同。见readback_verification.json。首次只读调用的URI参数错误已修正，无额外训练或账户。
